# ChronoQuant: Comprehensive Evaluation Suite

This notebook provides a rigorous, end-to-end evaluation of the ChronoQuant KV-Cache compression architecture on the latest HuggingFace models.

**Evaluation Metrics:**
1. **Throughput (tok/sec)**: Baseline vs Compressed generation speed.
2. **Memory Footprint**: Cache size requirements and compression ratios.
3. **Perplexity (PPL)**: Language modeling degradation over Wikitext-2.
4. **Factual Retrieval**: Long-context Needle In A Haystack (NIAH) pass rates.

In [ ]:
from google.colab import drive
import sys
import os

# Mount Drive
drive.mount("/content/drive")
CHRONOQUANT_DIR = "/content/drive/MyDrive/chronoquant"
sys.path.append(CHRONOQUANT_DIR)

!pip install -q git+https://github.com/huggingface/transformers.git accelerate triton torch xformers bitsandbytes datasets

In [ ]:
import torch
import time
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers.models.qwen2.modeling_qwen2 as qwen2_modeling
from datasets import load_dataset

def apply_chronoquant_patch(model):
    _original_forward = qwen2_modeling.Qwen2Attention.forward
    
    def _chronoquant_forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False, **kwargs):
        bsz, q_len, _ = hidden_states.size()
        query_states = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        kv_seq_len = key_states.shape[-2]
        if past_key_value is not None:
            kv_seq_len += past_key_value.get_usable_length(kv_seq_len, self.layer_idx)
        cos, sin = self.rotary_emb(value_states, seq_len=kv_seq_len)
        query_states, key_states = qwen2_modeling.apply_rotary_pos_emb(query_states, key_states, cos, sin, position_ids)

        if past_key_value is not None:
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, {"queries": query_states})
            
            # Robust 4/3/2-Bit Mathematical Quantization
            D = self.head_dim
            n_4b = min(D, 64)
            n_3b = min(D - n_4b, 64)
            n_2b = max(0, D - n_4b - n_3b)
            
            k_parts = []
            if n_4b > 0:
                k_4b = key_states[..., :n_4b]
                s_4b = (k_4b.abs().max(dim=-1, keepdim=True)[0] / 7.0).clamp(min=1e-5)
                k_parts.append(torch.round(k_4b / s_4b).clamp(-7, 7) * s_4b)
            if n_3b > 0:
                k_3b = key_states[..., n_4b:n_4b+n_3b]
                s_3b = (k_3b.abs().max(dim=-1, keepdim=True)[0] / 3.0).clamp(min=1e-5)
                k_parts.append(torch.round(k_3b / s_3b).clamp(-3, 3) * s_3b)
            if n_2b > 0:
                k_2b = key_states[..., n_4b+n_3b:]
                s_2b = (k_2b.abs().max(dim=-1, keepdim=True)[0] / 1.0).clamp(min=1e-5)
                k_parts.append(torch.round(k_2b / s_2b).clamp(-1, 1) * s_2b)
            
            k_rec = torch.cat(k_parts, dim=-1)
            
            k_rec = torch.repeat_interleave(k_rec, dim=1, repeats=self.num_key_value_groups)
            v_rec = torch.repeat_interleave(value_states, dim=1, repeats=self.num_key_value_groups)
            attn_output = torch.nn.functional.scaled_dot_product_attention(query_states, k_rec, v_rec, attn_mask=attention_mask)
            attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, self.hidden_size)
            return self.o_proj(attn_output), None, past_key_value

        key_states = torch.repeat_interleave(key_states, dim=1, repeats=self.num_key_value_groups)
        value_states = torch.repeat_interleave(value_states, dim=1, repeats=self.num_key_value_groups)
        attn_output = torch.nn.functional.scaled_dot_product_attention(query_states, key_states, value_states, attn_mask=attention_mask)
        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, self.hidden_size)
        return self.o_proj(attn_output), None, past_key_value

    qwen2_modeling.Qwen2Attention.forward = _chronoquant_forward
    return _original_forward

def remove_patch(orig):
    qwen2_modeling.Qwen2Attention.forward = orig


In [ ]:
MODEL_ID = "unsloth/Qwen3.5-4B"
print(f"Downloading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.float16)


In [ ]:
def calculate_compression_ratio(model, seq_len=8192):
    cfg = model.config
    layers = cfg.num_hidden_layers
    kv_heads = cfg.num_key_value_heads
    head_dim = cfg.head_dim
    print(f"\n--- Compression Ratio Analysis (Context: {seq_len} tokens) ---")
    baseline_bytes = (layers * kv_heads * seq_len * head_dim * 2 * 2)
    n_4b = min(64, head_dim)
    n_3b = min(64, max(0, head_dim - n_4b))
    n_2b = max(0, head_dim - n_4b - n_3b)
    k_bytes = (n_4b * 0.5) + (n_3b * 0.375) + (n_2b * 0.25)
    v_bytes = head_dim * 2
    cq_bytes = layers * kv_heads * seq_len * (k_bytes + v_bytes)
    print(f"Baseline: {baseline_bytes/(1024**2):.2f} MB")
    print(f"ChronoQuant: {cq_bytes/(1024**2):.2f} MB")
    print(f"Ratio: {baseline_bytes/cq_bytes:.2f}x")
calculate_compression_ratio(model)

In [ ]:
def measure_speed(model, tokenizer, text, tokens=30):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**inputs, use_cache=True)
        pkvs = out.past_key_values
        nxt = out.logits[:, -1, :].argmax(dim=-1).unsqueeze(0)
        start = time.time()
        for _ in range(tokens):
            out = model(input_ids=nxt, past_key_values=pkvs, use_cache=True)
            pkvs = out.past_key_values
            nxt = out.logits[:, -1, :].argmax(dim=-1).unsqueeze(0)
            torch.cuda.synchronize()
        return tokens / (time.time() - start)
LONG_TEXT = "Quantum " * 400
print("\n--- TPS Baseline ---")
print(f"Speed: {measure_speed(model, tokenizer, LONG_TEXT):.2f}")
print("\n--- TPS ChronoQuant ---")
orig = apply_chronoquant_patch(model)
try: print(f"Speed: {measure_speed(model, tokenizer, LONG_TEXT):.2f}")
finally: remove_patch(orig)

In [ ]:
def get_ppl(model, tokenizer, data):
    enc = tokenizer("\n\n".join(data["text"][:15]), return_tensors="pt")
    nlls = []
    for i in range(0, enc.input_ids.size(1), 512):
        x = enc.input_ids[:, i:i+512].to(model.device)
        with torch.no_grad():
            nlls.append(model(x, labels=x).loss)
    return torch.exp(torch.stack(nlls).mean()).item()
data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
print("\n--- PPL Baseline ---")
print(f"PPL: {get_ppl(model, tokenizer, data):.4f}")
print("\n--- PPL ChronoQuant ---")
orig = apply_chronoquant_patch(model)
try: print(f"PPL: {get_ppl(model, tokenizer, data):.4f}")
finally: remove_patch(orig)

In [ ]:
def niah(model, tokenizer):
    h = "Atlantis was a city. " * 300
    n = "The passcode is 834927. "
    q = "\n\nQuestion: What is the passcode? Answer: "
    inp = tokenizer(h + n + h + q, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=10)
    res = tokenizer.decode(out[0][inp.input_ids.shape[1]:])
    print(f"Output: {res}")
    print("PASSED" if "834927" in res else "FAILED")
print("\n--- NIAH ChronoQuant ---")
orig = apply_chronoquant_patch(model)
try: niah(model, tokenizer)
finally: remove_patch(orig)

In [ ]:
import torch
import time
from triton_kernels import chronoquant_attention

def benchmark_triton_speedup():
    print("--- Fused Triton Hardware Benchmark ---")
    BATCH, HEADS, SEQ_LEN, HEAD_DIM = 1, 32, 16384, 128
    device = "cuda"

    # 1. Setup Data
    q = torch.randn((BATCH, HEADS, HEAD_DIM), device=device, dtype=torch.float32)
    v = torch.randn((BATCH, HEADS, SEQ_LEN, HEAD_DIM), device=device, dtype=torch.float32)
    r_inv = torch.eye(HEAD_DIM, device=device, dtype=torch.float32)

    # Simulate ChronoQuant compressed cache buffers
    class DummyCache:
        pass
    cache = DummyCache()
    cache.c_4b = torch.randint(0, 255, (BATCH, HEADS, SEQ_LEN, 32), device=device, dtype=torch.int32)
    # Fixed: Use uint8 for 0-255 range to avoid signed char overflow
    cache.c_3b = torch.randint(0, 255, (BATCH, HEADS, SEQ_LEN, 64), device=device, dtype=torch.uint8)
    cache.c_2b = torch.randint(0, 255, (BATCH, HEADS, SEQ_LEN, 16), device=device, dtype=torch.int32)
    cache.s_4b = torch.randn((BATCH, HEADS, SEQ_LEN), device=device, dtype=torch.float32)
    cache.s_3b = torch.randn((BATCH, HEADS, SEQ_LEN), device=device, dtype=torch.float32)
    cache.s_2b = torch.randn((BATCH, HEADS, SEQ_LEN), device=device, dtype=torch.float32)

    # 2. Benchmark Baseline (FP16 Attention simulation)
    k_fp16 = torch.randn((BATCH, HEADS, SEQ_LEN, HEAD_DIM), device=device, dtype=torch.float16)
    q_fp16 = q.to(torch.float16).unsqueeze(2)

    def baseline_step():
        scores = torch.matmul(q_fp16, k_fp16.transpose(-2, -1))
        return torch.matmul(torch.softmax(scores, dim=-1), v.to(torch.float16))

    # Warmup
    for _ in range(20): baseline_step()
    torch.cuda.synchronize()

    start = time.time()
    for _ in range(200): baseline_step()
    torch.cuda.synchronize()
    t_base = (time.time() - start) / 200

    # 3. Benchmark ChronoQuant Triton
    def chrono_step():
        return chronoquant_attention(q, r_inv, cache, v, SEQ_LEN)

    # Warmup
    for _ in range(20): chrono_step()
    torch.cuda.synchronize()

    start = time.time()
    for _ in range(200): chrono_step()
    torch.cuda.synchronize()
    t_cq = (time.time() - start) / 200

    print(f"Context Length       : {SEQ_LEN} tokens")
    print(f"FP16 Baseline        : {t_base * 1000:.3f} ms / token")
    print(f"ChronoQuant Triton   : {t_cq * 1000:.3f} ms / token")
    print(f"Hardware Speedup     : {t_base / t_cq:.2f}x")

benchmark_triton_speedup()